In [ ]:
# Import everything we need from SymPy, numpy, scipy, and matplotlib
import sympy as sym
import numpy as np
import scipy as sci
from sympy import Symbol, Matrix, Eq, Piecewise
from sympy import N, diff, simplify, sin, cos, solve, nsolve, solveset
from sympy import init_printing
import matplotlib.pyplot as plt
init_printing() # This function will make the outputs of SymPy look prettier and be easier to read

In [ ]:
# Define the wing parameters for a Cessna 172
S_w = 16.2             # Wing area in sq meters
AR_w = 7.52            # Wing aspect ratio
c_w = 1.49             # Wing mean chord in meters
r_w = 0.111            # Distance from CoM to center of lift of wing in meters
alpha_0_w = 6.36       # Wing airfoil lift slope (dcl / dalpha) in 1/radians
alpha_L0_w = -0.0436   # Wing airfoil 0 lift angle of attack in radian

# Define the horizontal stabilizer and elevator parameters for a Cessna 172
S_t = 2.00   # Horizontal stabilizer area in sq meters
AR_t = 6.32  # Horizontal stabilizer aspect ratio
S_e = 1.35   # Elevator area in sq meters
AR_e = 9.37  # Elevator aspect ratio
c_te = 0.942           # Combined horizontal stabilizer and elevator mean chord in meters
r_te = 4.56            # Distance from CoM to center of lift of horizontal stabilizer and elevator system in meters
alpha_0_te = 6.36      # Horizontal stabilizer and elevator airfoil lift slope (dcl / dalpha) in 1/radians
alpha_L0_te = 0.0436   # Horizontal stabilizer and elevator airfoil 0 lift angle of attack in radian

# Define the fuselage parameters for a Cessna 172
S_b = 5.59    # Effective drag area of fuselage in sq meters
cDf_b = 0.095 # Profile drag coefficient of fuselage (streamlined body at Re=6e6)

# Define other parameters for a Cessna 172
mass = 964            # Mass of plane in kg
I_yy = 2430           # Pitching moment of inertia in kg-m^2
eta = 0.7             # Effective efficiency of engine (engine efficiency time prop efficiency)
P_max = 1.34e5        # Maximum engine power in Watts
alpha_stall = 0.297   # Stall angle of attack for the wing and horizontal stab in radians
delta_up_max = 0.384  # Maximum upward (nose up) elevator deflection possible from trim control in radians 
delta_dwn_max = 0.332 # Maximum downward (nose down) elevator deflection possible from trim control in radians
v_s1 = 24.4           # Reference stall speed in clean configuration
v_n0 = 66.9           # Maximum structural cruise speed

# Define physical parameters
g = 9.81     # Acceleration due to gravity in m-s^{-2} at 2000 meters msl
rho = 1.01   # Density of STP air in kg-m^{-3} at 2000 meters msl
mu = 1.73e-5 # Dynamic viscosity of STP air in Pa-s at 2000 meters msl

In [ ]:
# Define the generalized coordinates (and their derivatives) and inputs
hdot = Symbol('hdot')         # Altitude rate
h = Symbol('h')               # Altitude 
udot_inf = Symbol('udot_inf') # Indicated airspeed rate
u_inf = Symbol('u_inf')       # Indicated airspeed 
alphadot = Symbol('alphadot') # Angle of attack rate
alpha = Symbol('alpha')       # Angle of attack
omegadot_theta = Symbol('omegadot_theta') # Pitch acceleration
omega_theta = Symbol('omega_theta')    # Pitch rate (state)
thetadot = Symbol('thetadot')          # Pitch angle (state derivative)
theta = Symbol('theta')                # Pitch angle
P = Symbol('P')         # Engine power
delta = Symbol('delta') # Elevator deflection

In [ ]:
# Define the lift of the wing (Nonsymmetric, elliptical lift distribution, thick, finite wing)
cL_w = ( alpha_0_w / (1.0 + alpha_0_w / (np.pi*AR_w)) ) * (alpha - alpha_L0_w)
L_w = 0.5*rho*S_w*cL_w*u_inf**2

In [ ]:
# Define the induced drag of the wing (Nonsymmetric, elliptical lift distribution, thick, finite wing)
cDi_w = ( (np.pi*AR_w*alpha_0_w**2) / (np.pi*AR_w + alpha_0_w)**2 ) * (alpha - alpha_L0_w)**2
Di_w = 0.5*rho*S_w*cDi_w*u_inf**2

In [ ]:
# Define the parasitic drag of the wing (Turbulent flow over thin, finite plate)
cDf_w = 0.074*(rho*c_w*u_inf / mu)**(-0.2)
Df_w = 0.5*rho*S_w*cDf_w*u_inf**2

In [ ]:
# Define the lift of the horizontal stabilizer (Symmetric, elliptical lift distribution, finite wing)
cL_t = ( alpha_0_te / (1.0 + alpha_0_te / (np.pi*AR_t)) ) * (alpha - alpha_L0_te)
L_t = 0.5*rho*S_t*cL_t*u_inf**2

In [ ]:
# Define the induced drag of the horizontal stabilizer (Symmetric, elliptical lift distribution, thick, finite wing)
cDi_t = ( (np.pi*AR_t*alpha_0_te**2) / (np.pi*AR_t + alpha_0_te)**2 ) * (alpha - alpha_L0_te)**2
Di_t = 0.5*rho*S_t*cDi_t*u_inf**2

In [ ]:
# Define the lift of the elevator (Symmetric, elliptical lift distribution, thick, finite wing)
cL_e = ( alpha_0_te / (1.0 + alpha_0_te / (np.pi*AR_e)) ) * (alpha - delta - alpha_L0_te)
L_e = 0.5*rho*S_e*cL_e*u_inf**2

In [ ]:
# Define the induced drag of the horizontal stabilizer (Symmetric, elliptical lift distribution, thick, finite wing)
cDi_e = ( (np.pi*AR_e*alpha_0_te**2) / (np.pi*AR_e + alpha_0_te)**2 ) * (alpha - delta - alpha_L0_te)**2
Di_e = 0.5*rho*S_e*cDi_e*u_inf**2

In [ ]:
# Define the parasitic drag of the horizontal stabilizer, elevator system (Turbulent flow over thin, finite plate)
cDf_te = 0.074*(rho*c_te*u_inf / mu)**(-0.2)
Df_te = 0.5*rho*(S_t+S_e)*cDf_te*u_inf**2

In [ ]:
# Define the profile drag of the fuselage (Streamlined body at Re=6e6)
Df_b = 0.5*rho*S_b*cDf_b*u_inf**2

In [ ]:
# Get the summed lifts and drags
L_tot = L_w + L_t + L_e
D_tot = Di_w + Di_t + Di_e + Df_w + Df_te + Df_b

In [ ]:
# Get the total downward force on the plane
F_z = mass*g + sin(theta-alpha)*D_tot - cos(theta-alpha)*L_tot - sin(theta)*(eta*P/u_inf)
F_z = simplify(F_z)
print('Total downward force:')
N(F_z, 3)

In [ ]:
# Get the total forward force on the plane
F_x = cos(theta)*(eta*P/u_inf) - cos(theta-alpha)*D_tot - sin(theta-alpha)*L_tot
F_x = simplify(F_x)
print('Total forward force:')
N(F_x, 3)

In [ ]:
##### Get the total upward pitching torque on the plane
tau_y = -cos(alpha)*(L_w*r_w + (L_t+L_e)*r_te) - sin(alpha)*((Di_w+Df_w)*r_w + (Di_t+Di_e+Df_te)*r_te)
tau_y = simplify(tau_y)
print('Total pitching torque:')
N(tau_y, 3)

In [ ]:
# Define the forward (u) and downward (v) velocities of the plane with respect to the generalized coords
u = u_inf * cos(theta - alpha)
v = -u_inf * sin(theta - alpha)

In [ ]:
# Perform the multivariable chain rule on v to get expression in term of generalized coords
# Use new expression to define first equation of motion according to Newton's second law
f1 = Eq(diff(v, u_inf)*udot_inf + diff(v, alpha)*alphadot + diff(v, theta)*omega_theta, F_z/mass)

# Perform the multivariable chain rule on u to get expression in term of generalized coords
# Use new expression to define second equation of motion according to Newton's second law
f2 = Eq(diff(u, u_inf)*udot_inf + diff(u, alpha)*alphadot + diff(u, theta)*omega_theta, F_x/mass)

# Apply Newton's second law with rotations to get third equation of motion
f3 = Eq(omegadot_theta, tau_y/I_yy)

# Use definition of altitude to get fourth equation of motion
f4 = Eq(hdot, -v)

# Use definition of pitch rate to get the fifth and final equation of motion
f5 = Eq(thetadot, omega_theta)

In [ ]:
# Build the system of equations of motion
system_of_eqns = (f1, f2, f3, f4, f5)
var = (hdot, udot_inf, alphadot, omegadot_theta, thetadot)

# Solve the equations of motion for the generalized coordinate first order terms
soln = solve(system_of_eqns, var)

# Build the solved equations of motion
f = simplify(Matrix([soln[hdot], soln[udot_inf], soln[alphadot], soln[omegadot_theta], soln[thetadot]]))

In [ ]:
# Simplify and print the equations of motion
print('Standard Form Equations of Motion:')
N(f,3)

In [ ]:
# Set the derivative states  to 0 to check what equilibrium positions look like
# Assuming we are at cruise, the pitch and angle of attack must be identical
sub_expr = {omega_theta : 0.0,  # To be stationary, the derivative of alpha must be 0
            theta : alpha,}      # Cruise condition

# Evaluate the equations of motion
equil_f = f.subs(sub_expr)
print('Equilibrium Condition (no separation):')
N(equil_f, 3)

In [ ]:
# Determine our optimal cruise conditions for best endurance, best range, and best speed
u_infs = np.linspace(v_s1, v_n0, 50)
cruise_conds = []
for u_inf_c in u_infs:
    # Evaluate the equations of motion at the given cruise condition
    # "c" is a common abbreviation used in aerospace
    # it stands for "cruise"
    f_c = f.subs({omega_theta : 0.0,
                  theta : alpha,
                  u_inf : u_inf_c})
    
    # Solve for cruise angle of attack, elevator deflection, and engine power settings
    # given the above defined cruise conditions
    soln = nsolve((f_c[1], f_c[2], f_c[3]),
                  (alpha, delta, P), 
                  (0.0, 0.0, mass*g*u_inf_c/10.0))
    alpha_c = float(soln[0])
    delta_c = float(soln[1])
    P_c = float(soln[2])
    
    # Determine if the solved cruise condition is consistent with system limitations
    is_consistent = (abs(alpha_c) <= alpha_stall, 
                     delta_c <= delta_up_max, 
                     delta_c >= -delta_dwn_max, 
                     P_c >= 0,
                     P_c <= P_max)
    if not all(is_consistent):
        continue
    
    # Calculate the lift on drag (maximize to maximize endurance)
    # Calculate the drag on velocity (minimize to maximize range)
    sub = {u_inf:u_inf_c, alpha:alpha_c, delta:delta_c}
    L_c = L_tot.subs(sub)
    D_c = D_tot.subs(sub)
    L_on_D = L_c / D_c 
    D_on_u_inf = D_c / u_inf_c

    # Collect the cruise data
    cruise_conds.append((u_inf_c, alpha_c, delta_c, P_c, L_on_D, D_on_u_inf))

# Format the collected cruise data
cruise_conds = np.array(cruise_conds, dtype=float)

In [ ]:
# Calculate the cruise conditions with the highest endurance 
# be is a common abbreviation used in airplanes 
# it stands for "best endurance"
be = np.argmax(cruise_conds[:,4])
u_inf_be = cruise_conds[be,0]
alpha_be = cruise_conds[be,1]
delta_be = cruise_conds[be,2]
P_be = cruise_conds[be,3]

# Calculate the cruise condtions with the highest range
# br is a common abbreviation used in airplanes 
# it stands for "best range"
br = np.argmin(cruise_conds[:,5])
u_inf_br = cruise_conds[br,0]
alpha_br = cruise_conds[br,1]
delta_br = cruise_conds[br,2]
P_br = cruise_conds[br,3]

# Calculate the cruise condtions with the highest speed
# n0 is a common abbreviation used in airplanes 
# it indicates the maximum structural cruising speed conditions
u_inf_n0 = cruise_conds[-1,0]
alpha_n0 = cruise_conds[-1,1]
delta_n0 = cruise_conds[-1,2]
P_n0 = cruise_conds[-1,3]

In [ ]:
# Our chosen equilibrium point (best range)
equil_br = {u_inf : u_inf_br,
            alpha : alpha_br,
            omega_theta : 0.0,
            theta : alpha_br,
            delta : delta_br,
            P : P_br, }

# Evaluate the equations of motion to ensure all are 0
print('EoM solved at best range condition:')
N(f.subs(equil_br), 3)

In [ ]:
# Define the nonlinear state vector
m = [h, u_inf, alpha, omega_theta, theta]

# Take the jacobian of f with respect to the nonlinear state vector
A_br = f.jacobian(m)

# Evaluate A at our selected equilibrium point
A_br = A_br.subs(equil_br)
A_br = np.array(A_br, dtype=float)
A_br = np.round(A_br, 15)

# Print A
print('A at best range:')
N(Matrix(A_br),3)

In [ ]:
# Define the nonlinear input vector
n = [delta, P]

# Take the jacobian of f with respect to the nonlinear input vector
B_br = f.jacobian(n)

# Evaluate B at our selected equilibrium point
B_br = B_br.subs(equil_br)
B_br = np.array(B_br, dtype=float)
B_br = np.round(B_br, 15)

# Print B
print('B at best range:')
N(Matrix(B_br),3)

In [ ]:
# Calculate the controllability matrix
ctrb_br = tuple(np.linalg.matrix_power(A_br, i) @ B_br for i in range(len(m)))
ctrb_br = np.hstack(ctrb_br).astype(float)
print('Controllability Matix at best range:')
N(Matrix(ctrb_br), 3)

In [ ]:
# Determine controllability by comparing the number of nonzero single values to the number of states
s = sci.linalg.svdvals(ctrb_br)
n_nonzero_singular_vals = np.sum(~np.isclose(s, 0)) # Account for numerical errors with np.isclose
is_controllable = n_nonzero_singular_vals == len(m)
print(f"Is Controllable at best range: {is_controllable}")

In [ ]:
# Define the maximum allowable values for each linear state
#                 h     u_inf  alpha            omega_theta       theta                            
z_max = np.array([25.0, 10.0,  np.deg2rad(8.0), np.deg2rad(10.0), np.deg2rad(15.0)])

# Use Bryson's rule to get Q
Q = np.diag( 1.0 / (z_max**2) )

# Define the maximum allowable values for each linear input
#                 delta             P                    
n_max = np.array([np.deg2rad(13.0), 2.8e4])

# Use Bryson's rule to get R
R = np.diag( 1.0 / (n_max**2) )

In [ ]:
# Solve the CARE and calculate the gain matrix
care_br = sci.linalg.solve_continuous_are(A_br, B_br, Q, R)
K_br = np.linalg.inv(R) @ B_br.T @ care_br

# Print the results
print('Control gains at best range:')
N(Matrix(K_br),3)

In [ ]:
def controller(state, h_des):
    """
    The controller function. Given some state information and a target altitude, calculates the elevator deflection and
    engine power to (hopefully) cruise at the desired condition.

    Parameters
    ----------
    state : dictionary of floats with the following keys
        h : float
            The altitude in meters
        u_inf : float
            The indicated airspeed in meters/second
        alpha : float
            The angle of attack in radians
        omega_theta : float
            The pitch rate in radians/second
        theta : float
            The pitch angle in radians
    h_des : float
        The target altitude in meters

    Returns
    -------
    delta : float
        The elevator deflection in radians
    P : float
        The engine power in Watts
    """
    # Define the equilibrium nonlinear state vector, equilibrium nonlinear
    # input vector, and desired linear state vector
    m_e = np.array([0.0, u_inf_br, alpha_br, 0.0, alpha_br])
    n_e = np.array([delta_br, P_br])
    x_des = np.array([h_des, 0.0, 0.0, 0.0, 0.0])

    # Build the control gain matrix
    K = K_br

    # Build the nonlinear state vector
    m = np.array([state['h'],
                  state['u_inf'],
                  state['alpha'],
                  state['omega_theta'],
                  state['theta'],])
    
    # Build the linear state vector
    x = m - m_e

    # Build the reference tracking linear state vector
    z = x - x_des

    # Apply the feedback control law with our selected gain matrix to
    # get the linear input vector
    u = -K@z

    # Convert the linear input vector into the nonlinear input vector
    n = u + n_e

    # Return the nonlinear torque as a scalar
    delta = n[0]
    P = n[1]
    return (delta, P)

In [ ]:
from pitch_ap import run
data = run(controller, 2)

In [ ]:
%matplotlib inline

fig, axs = plt.subplots(4, sharex = True)

axs[0].plot(data['time'], 3.28084*data['h'], label='Altitude', c='b', lw=1.5)
axs[0].plot(data['time'], 3.28084*data['h_des'], label='Des Altitude', c='r', ls='--', lw=1.5)
axs[0].legend(bbox_to_anchor=(0., 0.5, 2.325, 0.), loc='center',)
axs[0].set_ylabel('[feet]')

axs[1].plot(data['time'], 1.94384*data['u_inf'], label='Airspeed', c='b', lw=1.5)
axs[1].axhline(1.94384*u_inf_br, 0.0455, 0.955, label='Trim Cond', c='r', ls='--', lw=1.5)
axs[1].legend(bbox_to_anchor=(0., 0.5, 2.325, 0.), loc='center',)
axs[1].set_ylabel('[kias]')

axs[2].plot(data['time'], np.rad2deg(data['alpha']), label='Angle of Attack', c='b', lw=1.5)
axs[2].axhline(np.rad2deg(alpha_br), 0.0455, 0.955, label='Trim Cond', c='r', ls='--', lw=1.5)
axs[2].legend(bbox_to_anchor=(0., 0.5, 2.325, 0.), loc='center',)
axs[2].set_ylabel('[degrees]')

axs[3].plot(data['time'], np.rad2deg(data['theta']), label='Pitch', c='b', lw=1.5)
axs[3].axhline(np.rad2deg(alpha_br), 0.0455, 0.955, label='Trim Cond', c='r', ls='--', lw=1.5)
axs[3].legend(bbox_to_anchor=(0., 0.5, 2.325, 0.), loc='center',)
axs[3].set_xlabel('[seconds]')
axs[3].set_ylabel('[degrees]')
axs[3].set_xlim(min(data['time']) - 0.05*(max(data['time'])-min(data['time'])),
                max(data['time']) + 0.05*(max(data['time'])-min(data['time'])))

plt.show()

In [ ]:
fig, axs = plt.subplots(2, sharex = True)

axs[0].plot(data['time'], np.rad2deg(data['delta']), label='Elevator Ang', c='b', lw=1.5)
axs[0].axhline(np.rad2deg(delta_br), 0.0455, 0.955, label='Trim Cond', c='r', ls='--', lw=1.5)
axs[0].legend(bbox_to_anchor=(0., 0.5, 2.325, 0.), loc='center',)
axs[0].set_ylabel('[degrees]')

axs[1].plot(data['time'], 0.00134102*data['P'], label='Engine Power', c='b', lw=1.5)
axs[1].axhline(0.00134102*P_br, 0.0455, 0.955, label='Trim Cond', c='r', ls='--', lw=1.5)
axs[1].legend(bbox_to_anchor=(0., 0.5, 2.325, 0.), loc='center',)
axs[1].set_xlabel('[seconds]')
axs[1].set_ylabel('[horse power]')
axs[1].set_xlim(min(data['time']) - 0.05*(max(data['time'])-min(data['time'])),
                max(data['time']) + 0.05*(max(data['time'])-min(data['time'])))

plt.show()